In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("titanic_train.csv")

print("Rows and columns:", df.shape)
print("Missing ages:", df["Age"].isna().sum())
df[["Pclass", "Sex", "Age", "Fare", "Survived"]].head()

Rows and columns: (891, 12)
Missing ages: 177


,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [2]:
X = df[["Pclass", "Sex", "Age", "Fare"]].copy()
y = df["Survived"]

X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

median_age = X_train["Age"].median()

X_train["Age"] = X_train["Age"].fillna(median_age)
X_test["Age"] = X_test["Age"].fillna(median_age)

print("Features:", X.columns.tolist())
print("Training passengers:", len(X_train))
print("Testing passengers:", len(X_test))
print("Median age used:", median_age)
print("Missing values remaining:", X_train.isna().sum().sum() + X_test.isna().sum().sum())

Features: ['Pclass', 'Sex', 'Age', 'Fare']
Training passengers: 623
Testing passengers: 268
Median age used: 28.0
Missing values remaining: 0


For this task, I chose the options of passenger class, sex, age, and fare to predict whether or not someone survived the Titanic. Also, dyue to the fact that some passengers didn't have their age as information, I used the median age of 28 from the training data to fill in the missing values instead of flat out removing those passangers.

In [3]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.2%}")

Test accuracy: 80.60%


In [4]:
errors = X_test.copy()

errors["Actual"] = y_test
errors["Predicted"] = predictions
errors["Wrong"] = errors["Actual"] != errors["Predicted"]

print("Total incorrect predictions:", errors["Wrong"].sum())

print("\nErrors by passenger class:")
print(errors.groupby("Pclass")["Wrong"].agg(["sum", "mean"]))

print("\nErrors by sex:")
print(errors.groupby("Sex")["Wrong"].agg(["sum", "mean"]))

Total incorrect predictions: 52

Errors by passenger class:
        sum      mean
Pclass               
1        20  0.259740
2         7  0.134615
3        25  0.179856

Errors by sex:
     sum      mean
Sex               
0     34  0.203593
1     18  0.178218


From the results, what I noticed was the fact that the model itself made the most incorrect predictions with 25 errors for third-class passengers, and with 34 errors for the male passengers. 

GitHub Repository

https://github.com/leonhong2007/cs82a-portfolio/tree/main/module7